In [ ]:
import os, sys, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import kagglehub
import glob
from tqdm import tqdm
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# custom_losses.py 경로 추가 (로컬)
sys.path.insert(0, '/root/inbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

# DRIVE 데이터셋 로드 (kagglehub 캐시 활용)
path = kagglehub.dataset_download("andrewmvd/drive-digital-retinal-images-for-vessel-extraction")
DATA_DIR = path

all_tifs = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.tif"), recursive=True))
all_gifs = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.gif"), recursive=True))

# manual1만 사용 (각 이미지에 manual1/manual2 두 개의 어노테이션 존재)
train_img_paths  = sorted([p for p in all_tifs if 'training' in p.lower()])
train_mask_paths = sorted([p for p in all_gifs if 'training' in p.lower() and 'manual1' in p.lower()])
# DRIVE test set에는 vessel GT(1st_manual)가 없으므로 train에서 val split 사용
test_img_paths   = sorted([p for p in all_tifs if 'test' in p.lower()])
test_mask_paths  = []  # GT 없음 — 평가는 val set으로 대체

# Train / Val 분리
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    train_img_paths, train_mask_paths, test_size=0.2, random_state=42)

# ── 결과 저장 경로 ───────────────────────────────────────────────────────────
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Train {len(train_img_paths)} | Val {len(val_img_paths)} | Test {len(test_img_paths)} (GT 없음)")


In [5]:
class RetinalPatchDataset(Dataset):
    def __init__(self, img_paths, mask_paths, patch_size=256, samples_per_epoch=500, transform=None):
        self.patch_size = patch_size
        self.samples_per_epoch = samples_per_epoch
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        self.images, self.masks = [], []
        for img_p, mask_p in zip(img_paths, mask_paths):
            img = cv2.imread(img_p)
            if img is None:
                continue
            enhanced = self.clahe.apply(img[:, :, 1])
            self.images.append(cv2.merge([enhanced] * 3))
            mask = np.array(Image.open(mask_p).convert('L'))  # PIL로 GIF 읽기
            self.masks.append((mask > 127).astype(np.uint8))

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, idx):
        i = random.randint(0, len(self.images) - 1)
        img, mask = self.images[i], self.masks[i]
        h, w = img.shape[:2]
        if random.random() > 0.5:
            vy, vx = np.where(mask == 1)
            if len(vy) > 0:
                ii = random.randint(0, len(vy) - 1)
                y = min(max(vy[ii] - self.patch_size // 2, 0), h - self.patch_size)
                x = min(max(vx[ii] - self.patch_size // 2, 0), w - self.patch_size)
            else:
                y, x = random.randint(0, h - self.patch_size), random.randint(0, w - self.patch_size)
        else:
            y, x = random.randint(0, h - self.patch_size), random.randint(0, w - self.patch_size)
        ip = img[y:y + self.patch_size, x:x + self.patch_size]
        mp = mask[y:y + self.patch_size, x:x + self.patch_size]
        if self.transform:
            aug = self.transform(image=ip, mask=mp)
            ip, mp = aug['image'], aug['mask']
        return ip, mp.long()

train_tf = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
                       A.RandomRotate90(p=0.5), A.Normalize(), ToTensorV2()])
val_tf   = A.Compose([A.Normalize(), ToTensorV2()])

train_loader = DataLoader(RetinalPatchDataset(train_img_paths, train_mask_paths,
                           transform=train_tf, samples_per_epoch=500),
                           batch_size=8, shuffle=True)
val_loader   = DataLoader(RetinalPatchDataset(val_img_paths, val_mask_paths,
                           transform=val_tf,   samples_per_epoch=100),
                           batch_size=8, shuffle=False)

# 클래스 비율 계산
print("클래스 비율 계산 중...")
bg, fg = 0, 0
for p in train_mask_paths:
    m = np.array(Image.open(p).convert('L'))
    fg += int((m > 127).sum())
    bg += int((m <= 127).sum())
class_counts = [bg, fg]
print(f"BG: {bg:,}  FG(vessel): {fg:,}  Ratio: {bg/fg:.1f}:1")


클래스 비율 계산 중...
BG: 4,828,573  FG(vessel): 450,787  Ratio: 10.7:1


In [6]:
# ── 1채널 → 2채널 대칭 logit 변환 (핵심 버그 수정) ─────────────────────────
# 기존: [zeros, p]  → 배경 logit 0 고정, 학습 불안정
# 수정: [-p, p]    → 대칭 logit, sigmoid(p)와 수학적으로 동일
def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

# 검증 Dice 계산
def compute_val_dice(model, loader):
    model.eval()
    dice_sum = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            res   = preds[-1] if isinstance(preds, (list, tuple)) else preds
            res   = F.interpolate(res, size=masks.shape[1:], mode='bilinear', align_corners=True)
            prob  = torch.sigmoid(res).squeeze(1)
            pred  = (prob > 0.5).long()
            inter = (pred.float() * masks.float()).sum()
            union = pred.float().sum() + masks.float().sum()
            dice_sum += (2. * inter / (union + 1e-8)).item() if union > 0 else 1.0
    return dice_sum / len(loader)

print("유틸리티 함수 정의 완료")


유틸리티 함수 정의 완료


In [7]:
# PraNet (이미 /tmp/PraNet에 클론+패치 완료 전제)
pranet_lib = '/tmp/PraNet/lib'
if pranet_lib not in sys.path:
    sys.path.insert(0, pranet_lib)

# 미클론 시 자동 클론+패치
if not os.path.exists('/tmp/PraNet'):
    os.system('git clone https://github.com/DengPingFan/PraNet.git /tmp/PraNet')
    # 상대 import 패치
    target = '/tmp/PraNet/lib/PraNet_Res2Net.py'
    with open(target) as f: code = f.read()
    if 'from .Res2Net_v1b' in code:
        with open(target, 'w') as f: f.write(code.replace('from .Res2Net_v1b', 'from Res2Net_v1b'))
    # Res2Net 가중치 경로 패치
    import urllib.request
    wp = '/tmp/PraNet/models/res2net50_v1b_26w_4s-3cf99910.pth'
    os.makedirs('/tmp/PraNet/models', exist_ok=True)
    if not os.path.exists(wp):
        urllib.request.urlretrieve(
            'https://shanghuagao.oss-cn-beijing.aliyuncs.com/res2net/res2net50_v1b_26w_4s-3cf99910.pth', wp)
    r2n = '/tmp/PraNet/lib/Res2Net_v1b.py'
    with open(r2n) as f: code = f.read()
    hc = '/media/nercms/NERCMS/GepengJi/Medical_Seqmentation/CRANet/models/res2net50_v1b_26w_4s-3cf99910.pth'
    if hc in code:
        with open(r2n, 'w') as f: f.write(code.replace(hc, wp))

for k in [k for k in sys.modules if 'Res2Net' in k or 'PraNet_Res2Net' in k]:
    del sys.modules[k]
from PraNet_Res2Net import PraNet

# IterNet (BatchNorm 적용)
class IterNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.out  = nn.Conv2d(64, 1, 1)
        self.iter = nn.Sequential(
            nn.Conv2d(65, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 1, 1))
    def forward(self, x):
        feat = self.base(x)
        out1 = self.out(feat)
        return [out1, self.iter(torch.cat([feat, out1], dim=1))]

print("모델 정의 완료: IterNet, PraNet")


모델 정의 완료: IterNet, PraNet


In [8]:
def train_model(model_class, loss_name, epochs=25, lr=1e-4):
    model     = model_class().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = get_loss_function(loss_name, class_counts=class_counts)
    name      = f"{model_class.__name__}+{loss_name}"
    print(f"\n{'='*55}\n{name}  (Epochs={epochs})\n{'='*55}")

    history   = {'loss': [], 'val_dice': []}
    best_dice = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for imgs, masks in tqdm(train_loader, desc=f"Ep{epoch+1:02d}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss  = 0
            if isinstance(preds, (list, tuple)):
                for p in preds:
                    p    = F.interpolate(p, size=masks.shape[1:], mode='bilinear', align_corners=True)
                    loss += criterion(to_2ch_logits(p), masks)
            else:
                p    = F.interpolate(preds, size=masks.shape[1:], mode='bilinear', align_corners=True)
                loss = criterion(to_2ch_logits(p), masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_loss = train_loss / len(train_loader)
        val_dice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)

        print(f"Ep{epoch+1:02d} | Loss:{avg_loss:.4f} | ValDice:{val_dice:.4f}", end="")
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f'/tmp/best_{name}.pth')
            print("  ← Best!", end="")
        print()

    model.load_state_dict(torch.load(f'/tmp/best_{name}.pth', weights_only=True))
    print(f"최고 Val Dice: {best_dice:.4f}")
    return model, history, best_dice

# IterNet + 4가지 Loss 비교
all_results = {}
for loss_name in ['ce_dice', 'wce_dice', 'lwce_dice', 'plwce_dice']:
    m, h, b = train_model(IterNet, loss_name, epochs=25)
    all_results[f'IterNet+{loss_name}'] = {'model': m, 'history': h, 'best_dice': b}

print("\n[IterNet Loss 비교 요약]")
for k, v in all_results.items():
    print(f"  {k}: Best Val Dice = {v['best_dice']:.4f}")



IterNet+ce_dice  (Epochs=25)


Ep01 | Loss:1.1641 | ValDice:0.6792  ← Best!


Ep02 | Loss:0.9351 | ValDice:0.7153  ← Best!


Ep03 | Loss:0.8409 | ValDice:0.7439  ← Best!


Ep04 | Loss:0.7732 | ValDice:0.7417


...

Ep21 | Loss:0.5712 | ValDice:0.7819  ← Best!


Ep22 | Loss:0.5658 | ValDice:0.7759


Ep23 | Loss:0.5631 | ValDice:0.7566


Ep24 | Loss:0.5593 | ValDice:0.7645


Ep25 | Loss:0.5562 | ValDice:0.7659
최고 Val Dice: 0.7819

[IterNet Loss 비교 요약]
  IterNet+ce_dice: Best Val Dice = 0.7682
  IterNet+wce_dice: Best Val Dice = 0.7692
  IterNet+lwce_dice: Best Val Dice = 0.7828
  IterNet+plwce_dice: Best Val Dice = 0.7819


In [14]:
def fast_predict(model, img_path):
    model.eval()
    img     = cv2.imread(img_path)
    h0, w0  = img.shape[:2]
    clahe   = cv2.createCLAHE(clipLimit=2.0)
    green   = clahe.apply(img[:, :, 1])
    resized = cv2.resize(cv2.merge([green] * 3), (256, 256))
    tf      = A.Compose([A.Normalize(), ToTensorV2()])
    tensor  = tf(image=resized)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        preds = model(tensor)
        res   = preds[-1] if isinstance(preds, (list, tuple)) else preds
        res   = F.interpolate(res, size=(h0, w0), mode='bilinear', align_corners=True)
        return torch.sigmoid(res).squeeze().cpu().numpy()

best_key   = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_key]['model']
print(f"시각화 모델: {best_key}")

# DRIVE test set에 vessel GT 없음 → val set으로 시각화
vis_imgs  = val_img_paths[:3]
vis_masks = val_mask_paths[:3]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(len(vis_imgs)):
    orig = cv2.cvtColor(cv2.imread(vis_imgs[i]), cv2.COLOR_BGR2RGB)
    gt   = np.array(Image.open(vis_masks[i]).convert('L')) / 255.0
    prob = fast_predict(best_model, vis_imgs[i])
    prob_resized = cv2.resize(prob, (gt.shape[1], gt.shape[0]))
    pred = (prob_resized > 0.5).astype(np.uint8)

    axes[i,0].imshow(orig);                    axes[i,0].set_title("Input Fundus"); axes[i,0].axis('off')
    axes[i,1].imshow(gt,   cmap='gray');        axes[i,1].set_title("Ground Truth"); axes[i,1].axis('off')
    axes[i,2].imshow(prob_resized, cmap='jet'); axes[i,2].set_title("Prob Map");     axes[i,2].axis('off')
    axes[i,3].imshow(pred, cmap='gray');        axes[i,3].set_title(f"Pred ({best_key})"); axes[i,3].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'retinal_visualization.png'), dpi=100)
plt.show()
print(f"시각화 저장: {os.path.join(RESULTS_DIR, 'retinal_visualization.png')}")


시각화 모델: IterNet+lwce_dice
시각화 저장: /tmp/retinal_visualization.png


In [15]:
from sklearn.metrics import roc_auc_score, confusion_matrix

def evaluate_model(model, img_paths, mask_paths, name):
    dices, sens, spes, aucs = [], [], [], []
    for i in range(len(img_paths)):
        gt   = (np.array(Image.open(mask_paths[i]).convert('L')) > 127).astype(np.uint8)
        prob = fast_predict(model, img_paths[i])
        prob = cv2.resize(prob, (gt.shape[1], gt.shape[0]))
        pred = (prob > 0.5).astype(np.uint8)
        try:
            aucs.append(roc_auc_score(gt.flatten(), prob.flatten()))
        except:
            aucs.append(0.5)
        tn, fp, fn, tp = confusion_matrix(gt.flatten(), pred.flatten(), labels=[0,1]).ravel()
        dices.append((2.*tp) / (2.*tp + fp + fn + 1e-8))
        sens.append(tp / (tp + fn + 1e-8))
        spes.append(tn / (tn + fp + 1e-8))
    return dict(Dice=np.mean(dices), Sens=np.mean(sens), Spec=np.mean(spes), AUC=np.mean(aucs))

import json
# DRIVE test set에 vessel GT 없음 → val set으로 평가
print("\n[전체 모델 종합 평가 (Val set — DRIVE test에 GT 없음)]")
print(f"{'Model':<28} {'Dice':>6} {'Sens':>6} {'Spec':>6} {'AUC':>6}")
print("-" * 52)
final_results = {}
for key, val in all_results.items():
    r = evaluate_model(val['model'], val_img_paths, val_mask_paths, key)
    final_results[key] = {k: float(v) for k, v in r.items()}
    print(f"{key:<28} {r['Dice']:>6.4f} {r['Sens']:>6.4f} {r['Spec']:>6.4f} {r['AUC']:>6.4f}")

with open(os.path.join(RESULTS_DIR, 'retinal_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)
print(f"\n결과 저장: {os.path.join(RESULTS_DIR, 'retinal_results.json')}")



[전체 모델 종합 평가 (Val set — DRIVE test에 GT 없음)]
Model                          Dice   Sens   Spec    AUC
----------------------------------------------------
IterNet+ce_dice              0.6986 0.7059 0.9686 0.9477
IterNet+wce_dice             0.6463 0.8226 0.9284 0.9532
IterNet+lwce_dice            0.7033 0.7171 0.9679 0.9502
IterNet+plwce_dice           0.6952 0.7483 0.9600 0.9493

결과 저장: /tmp/retinal_results.json
